# Spectral Axes: Velocity Frames and Doppler Conventions

In this notebook we show how to change the spectral axis to represent different rest frames and Doppler conventions.  In addition, a spectrum is normally
associated with a redshift/radial velocity, which can also be modified. We also show how to save a spectrum in
frequency and velocity space.

Supported **doppler conventions**: (see also https://www.gb.nrao.edu/~fghigo/gbtdoc/doppler.html)
1. radio
2. relativistic
3. optical

Some of the supported **rest frames**:
1. itrs - topocentric
2. icrs - barycentric
3. gcrs - geocentric
4. hcrs - heliocentric
5. lsrk, lsrd  - Local Standard of Rest (Kinematic or Dynamic)

This dysh code dealing with this relies heavily on [specutils](https://specutils.readthedocs.io/). To note is that our `Spectrum` class is inherited from the (now) namesake `Spectrum` class in specutils. Internally we still call that class in specutils `Spectrum1D`.

Although the `Spectrum.plot()` can plot a spectrum in different frames with different conventions, it does not modify the underlying data and meta-data in the spectrum.  To make these persistent (e.g. necessary when writing a spectrum) there is both an in-place and copy operation to modify frame and convention. Here's a dysh command summary that we will cover in this notebook, leaving out the values and arguments that are not relevant:

Note that *frame* and *convention* have confusing keywords, this ought to be refactored.

```
       ta.plot(vel_frame=, doppler_convention=)

       ta.velocity_axis_to(toframe=, doppler_convention=)

       ta.set_frame()
       ta.set_convention()

       ta1 = ta.with_frame()
       ta2 = ta1.with_velocity_convention()

       ta3 = ta.with_spectral_axis_unit("km/s")

       ta.set_redshift.to()
       ta.set_radial_velocity_to()
       ta.shift_spectrum_to(redshift=)
       ta.shift_spectrum_to(radial_velocity=)

       ta.rest_value =

       # no setters for these
       ta.redshift
       ta.radial_velocity
```


## Loading Modules
We start by loading the modules we will use for the data reduction. 


In [1]:
# These modules are required for working with the data.
from dysh.fits.gbtfitsload import GBTFITSLoad
from dysh.log import init_logging
from astropy import units as u
from dysh.spectra.spectrum import Spectrum

# These modules are used for file I/O
from dysh.util.files import dysh_data
from pathlib import Path

## Setup
We start the dysh logging, so we get more information about what is happening.
This is only needed if working on a notebook.
If using the CLI through the ``dysh`` command, then logging is setup for you.

In [2]:
init_logging(2)

# also create a local "output" directory where temporary notebook files can be stored.
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)

## Data Retrieval

Download the example SDFITS data, if necessary.

In [3]:
filename = dysh_data(test="getps")

13:51:21.593 I Resolving test=getps -> AGBT05B_047_01/AGBT05B_047_01.raw.acs/


## Data Loading

In [4]:
sdfits = GBTFITSLoad(filename)
sdfits.summary()

SCAN,OBJECT,VELOCITY,PROC,PROCSEQN,RESTFREQ,DOPFREQ,# IF,# POL,# INT,# FEED,AZIMUTH,ELEVATION
51,NGC5291,4386.0,OnOff,1,1.420405,1.420405,1,2,11,1,198.3431,18.6427
52,NGC5291,4386.0,OnOff,2,1.420405,1.420405,1,2,11,1,198.9306,18.7872
53,NGC5291,4386.0,OnOff,1,1.420405,1.420405,1,2,11,1,199.3305,18.3561
54,NGC5291,4386.0,OnOff,2,1.420405,1.420405,1,2,11,1,199.9157,18.4927
55,NGC5291,4386.0,OnOff,1,1.420405,1.420405,1,2,11,1,200.3042,18.0575
56,NGC5291,4386.0,OnOff,2,1.420405,1.420405,1,2,11,1,200.8906,18.1860
57,NGC5291,4386.0,OnOff,1,1.420405,1.420405,1,2,11,1,202.3275,17.3853
58,NGC5291,4386.0,OnOff,2,1.420405,1.420405,1,2,11,1,202.9192,17.4949


More background on the GBT SDFITS files can be found on
https://dysh.readthedocs.io/en/latest/reference/sdfits_files/gbt_sdfits.html

## Data Reduction

Next we fetch and calibrate the position switched data. We will use this data to show how to change rest frames and Doppler conventions. All of these occur at the spectrum level, so we need to get a spectrum first. 
More details can be found in the 
[Position Switching](https://dysh.readthedocs.io/en/latest/users_guide/positionswitch.html)
notebook.

We use the time-averaged spectrum from scans 51/52:

In [5]:
ta = sdfits.getps(scan=51, ifnum=0, plnum=0, fdnum=0).timeaverage()

## Changing the x-axis of a `Spectrum` Plot
Note this changes the axis of the plot but does not affect the underlying Spectrum object.


### Default Rest Frame

The default plot uses the frequency frame and Doppler convention found in the SDFITS file.
In this case, that is topocentric frame (ITRS) and the optical convention. 

In [6]:
ta.plot();

Although the spectrum `ta` can be printed using `print`, just printing it as an object gives a lot more information

In [7]:
print(ta)
print("\n")
ta

Spectrum (length=32768)
Flux=[0.2448421  0.31819268 0.19866335 ... 0.57650381 0.2179878
      1.20767879] K,  mean=0.28071 K
Spectral Axis=[1.42481684e+09 1.42481531e+09 1.42481379e+09 ...
               1.37482142e+09 1.37481989e+09 1.37481836e+09] Hz,  mean=1399817601.06055 Hz




<Spectrum(flux=[0.24484210085049396 ... 1.2076787931163384] K (shape=(32768,), mean=0.28071 K); spectral_axis=<SpectralAxis 
   (observer: <ITRS Coordinate (obstime=2005-06-27T02:05:58.000, location=(0.0, 0.0, 0.0) km): (x, y, z) in m
                  (882593.9465029, -4924896.36541728, 3943748.74743984)
               (v_x, v_y, v_z) in km / s
                  (0., 0., 0.)>
    target: <SkyCoord (FK5: equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
                (206.85210758, -30.40701531, 1000000.)
             (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                (0., 0., 4386.)>
    observer to target (computed from above):
      radial_velocity=4410.070039086893 km / s
      redshift=0.014820217767934851
    doppler_rest=1420405000.0 Hz
    doppler_convention=optical)
  [1.42481684e+09 1.42481531e+09 1.42481379e+09 ... 1.37482142e+09
 1.37481989e+09 1.37481836e+09] Hz> (length=32768))>

A large amount of information is also stores in the `meta` data, which is a python dictionary associated with the Spectrum.
Here is how to access the meta data of a spectrum, where we use a python trick to make the dictionary shown in alphabetical order:

In [8]:
dict(sorted(ta.meta.items()))

{'AP_EFF': np.float64(0.7047451620556072),
 'AZIMUTH': 198.1588901992111,
 'BACKEND': 'Spectrometer',
 'BANDWID': 50000000.0,
 'BINTABLE': 0,
 'BUNIT': 'K',
 'CAL': 'F',
 'CALPOSITION': 'Unknown',
 'CALTYPE': 'LOW',
 'CDELT1': -1525.87890625,
 'CRPIX1': 16385.0,
 'CRVAL1': 1399816838.1210938,
 'CRVAL2': np.float64(206.85210757719534),
 'CRVAL3': np.float64(-30.407015310885345),
 'CRVAL4': -6,
 'CTYPE1': 'FREQ-OBS',
 'CTYPE2': 'RA',
 'CTYPE3': 'DEC',
 'CTYPE4': 'STOKES',
 'CUNIT1': 'Hz',
 'CUNIT2': 'deg',
 'CUNIT3': 'deg',
 'DATE': '2024-03-15T14:05:45',
 'DATE-OBS': '2005-06-27T02:05:58.00',
 'DOPFREQ': 1420405000.0,
 'DURATION': np.float64(55.5225),
 'ELEVATIO': 18.69449298449073,
 'EQUINOX': 2000.0,
 'EXPOSURE': np.float64(53.7157844463702),
 'EXTEND': True,
 'EXTNAME': 'SINGLE DISH',
 'FDNUM': 0,
 'FEED': 1,
 'FEEDEOFF': 0.0,
 'FEEDXOFF': 0.0,
 'FITSINDEX': 0,
 'FITSVER': '1.9',
 'FREQRES': 1846.3134765625,
 'FRONTEND': 'Rcvr1_2',
 'HDU': 1,
 'HUMIDITY': 0.754,
 'IFNUM': 0,
 'INSTRU

In [9]:
#print(ta.velocity)
#print(ta.radial_velocity)   # 4410.070039086893 

For the remainder of the notebook, we focus on the small section of the spectrum near 1.39 GHz where a random spike can be seen. After some trial and error this happens in channel 21921 where the peak value is 0.59789101 K.

Let's plot this zoomed spectrum in km/s so we can better see the effect of frames and conventions.

In [10]:
# pick a zoomed spectrum  (this cell is safe to re-execute)
if ta.nchan > 1000:
    print(f"Taking a zoomed spectrum")
    print(ta[21920:21923])   # print values around the little spike, peak is at 21921
    ta = ta[21850:22000]     # these are the 150 channels we will zoom into
else:
    print(f"Already had the zoomed spectrum of {ta.nchan} channels")
    print(ta[70:73])         # peak is at 71

ta.plot(xaxis_unit='km/s')
print(ta.velocity_axis_to("km/s")[71])     # 6256.475    velocity of the spike in this frame 
print(ta.frequency[71].value)              # 1.3913680466171938  GHz


Taking a zoomed spectrum
Spectrum (length=3)
Flux=[0.33822484 0.59789101 0.26609857] K,  mean=0.40074 K
Spectral Axis=[1.39136957e+09 1.39136805e+09 1.39136652e+09] Hz,  mean=1391368046.61719 Hz


6256.475163870069 km / s
1.3913680466171938


## Writing a spectrum 

The default `Spectrum.write()` uses its native units (Hz) to write a spectrum. Arguably more useful is the spectral axis in km/s. One way is to make a copy of the spectrum while changing the units, using 
[Spectrum.with_spectral_axis_unit()](https://specutils.readthedocs.io/en/stable/api/specutils.Spectrum.html#specutils.Spectrum.with_spectral_axis_unit)

But first a default write operation, with spectral axis in Hz:

In [11]:
def head(filename, nlines=3):
    """ emulate the unix `head` program
    """
    with open(filename, 'r') as file:
        for i, line in enumerate(file, 1):
            print(line.strip())
            if i == nlines:
                break

In [12]:
ta.write("ngc5291_spike.tab",format="ascii.commented_header", overwrite=True)  
head('ngc5291_spike.tab')

# spectral_axis flux uncertainty weight mask baseline
1391476384.0195374 0.34372934266958427 0.0 218.81991773344748 0 0.0
1391474858.1406312 0.27963123045821414 0.0 218.81991773344748 0 0.0


In [13]:
ta1 = ta.with_spectral_axis_unit("km/s")
ta1.write("ngc5291_spike.tab",format="ascii.commented_header", overwrite=True)  
head('ngc5291_spike.tab')

# spectral_axis flux uncertainty weight mask baseline
6232.6468425347 0.34372934266958427 0.0 218.75420976172921 0 0.0
6232.982426932423 0.27963123045821414 0.0 218.75420976172921 0 0.0


In [14]:
print(ta.velocity_axis_to("km/s")[71]) 

6256.475163870069 km / s


In [15]:
print(ta1.velocity_axis_to("km/s")[71]) 

6256.475163870069 km / s


In [16]:
# short cut to prevent something like ta1 to become
ta.with_frame("gcrs").with_spectral_axis_unit("km/s",velocity_convention="radio", rest_value=1420*u.MHz).write("ngc5291_spike.tab",format="ascii.commented_header", overwrite=True)

### Change Rest Frame
You can change the velocity frame by supplying one of the [built-in astropy coordinate frames](https://docs.astropy.org/en/stable/coordinates/index.html#built-in-frame-classes). These are specified by a string name.

For example, to plot in the barycentric frame use `"icrs"`. The change from topocentric to barycantric is small, about 25 km/s in this case.

In [17]:
ta.plot(vel_frame='icrs', xaxis_unit='km/s',grid=True);

Recall that the peak was at 6256.5 km/s in the topocentric frame, whereas in the barycentric frame it is more like 6231.9 km/s.

In addition to the `astropy` frame names, we also allow `'topo'` and `'topocentric'`.

In [18]:
ta.plot(vel_frame='topo', xaxis_unit='km/s');

### Doppler Convention 
One can also change the Doppler convention between `radio`, `optical`, and `relativistic`.
Here we also change the x-axis to velocity units and use LSRK frame.

In [19]:
ta.plot(vel_frame='lsrk', doppler_convention='radio', xaxis_unit='km/s');

Finally, if you plot velocity units on the x-axis with no `vel_frame` given, it will default to the 
frame decoded from the VELDEF keyword in the header if present. (PJT:  no, topo/opt)

In [20]:
ta.plot(xaxis_unit="km/s");

## Changing the Spectral Axis of the `Spectrum`.
There are two ways to accomplish this.  One returns a copy of the original Spectrum with the new spectral axis; 
the other changes the spectral axis in place.

The `with_frame()` and `with_velocity_convention()` are used when returning a copy

The `set_frame()` and `set_convention()` are used in place  [PJT: notice the asymmetric used of function names]  -> feature requesy

### A.   Return a copy of the spectrum using [Spectrum.with_frame](https://dysh.readthedocs.io/en/latest/reference/modules/dysh.spectra.html#dysh.spectra.spectrum.Spectrum.with_frame)


In [21]:
newspec = ta.with_frame('galactocentric')                              # OPTI-GAL, but the V value has not changed !!!
newspec.plot(xaxis_unit="km/s",doppler_convention='radio')
print(f"The new spectral axis frame is {newspec.velocity_frame}")

print(newspec.velocity_axis_to("km/s", doppler_convention="radio")[71]) # 6097.621(optical) 5976.071 (radio)

The new spectral axis frame is galactocentric
5976.071520424437 km / s


In [22]:
print(newspec.velocity_axis_to("km/s")[71])     # 6044.81949434711
print(newspec.frequency[71].value)              # 1.3913680466171938  

6097.621687333563 km / s
1.39209060568669


In [23]:
newspec2 = ta.with_frame('galactocentric').with_velocity_convention("radio")
#newspec2 = ta.with_velocity_convention("radio").with_frame('galactocentric')
newspec2.plot(xaxis_unit="km/s")
print(newspec2.velocity_axis_to("km/s")[71]) # 5976.071 (radio)

5976.071520424437 km / s


In [24]:
print(newspec2.velocity_axis_to("km/s")[71])     # 6044.81949434711
print(newspec2.frequency[71].value)              # 1.3913680466171938  

5976.071520424437 km / s
1.39209060568669


One can see that the spectral axis of the new spectrum is different. About 722 kHz.

In [25]:
ta.spectral_axis - newspec.spectral_axis


<Quantity [-722615.33079433, -722614.53838181, -722613.74596906,
           -722612.9535563 , -722612.16114378, -722611.36873102,
           -722610.5763185 , -722609.78390574, -722608.99149323,
           -722608.19908047, -722607.40666795, -722606.61425519,
           -722605.82184243, -722605.02942991, -722604.23701715,
           -722603.44460464, -722602.65219188, -722601.85977936,
           -722601.0673666 , -722600.27495408, -722599.48254132,
           -722598.69012856, -722597.89771605, -722597.10530329,
           -722596.31289077, -722595.52047801, -722594.72806549,
           -722593.93565273, -722593.14324021, -722592.35082746,
           -722591.55841494, -722590.76600218, -722589.97358942,
           -722589.1811769 , -722588.38876414, -722587.59635162,
           -722586.80393887, -722586.01152635, -722585.21911359,
           -722584.42670107, -722583.63428831, -722582.84187555,
           -722582.04946303, -722581.25705028, -722580.46463776,
           -722579.672225

### B.   Change the spectral axis in place using [Spectrum.set_frame](https://dysh.readthedocs.io/en/latest/reference/modules/dysh.spectra.html#dysh.spectra.spectrum.Spectrum.set_frame)


In [28]:
print(ta.velocity_axis_to("km/s")[71]) # 6256.475
sa = ta.spectral_axis
ta.set_frame('gcrs')
print(f"Changed spectral axis frame to  {ta.velocity_frame}")
print(ta.velocity_axis_to("km/s")[71]) # 6256.365

6256.365280719197 km / s
Changed spectral axis frame to  gcrs
6256.365280719197 km / s


In [29]:
for frame in ['itrs', 'gcrs', 'icrs', 'gcrs', 'hcrs', 'lsr', 'lsrk', 'lsrd']:
    ta.set_frame(frame)
    v = ta.velocity_axis_to("km/s")[71]
    print(frame,v)

ValueError: For topographic or ITRS coordinates, you must supply a full astropy Coordinate instance.

The difference between GCRS (geocentric) and ITRS (topocentric) is very small, a mere 0.110 km/s

In [30]:
ta

<Spectrum(flux=[0.34372934266958427 ... 0.30729395889669986] K (shape=(150,), mean=0.28094 K); spectral_axis=<SpectralAxis 
   (observer: <GCRS Coordinate (obstime=2005-06-27T02:05:58.000, obsgeoloc=(0., 0., 0.) m, obsgeovel=(0., 0., 0.) m / s): (x, y, z) in m
                  (-3418483.37116048, -3651331.51587568, 3945691.32972075)
               (v_x, v_y, v_z) in km / s
                  (-4.3110922e-08, 9.51169059e-08, 6.55464828e-08)>
    target: <FK5 Coordinate (equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
                (206.85210758, -30.40701531, 1000000.)
             (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                (0., 0., 4386.)>
    observer to target (computed from above):
      radial_velocity=4409.962402228085 km / s
      redshift=0.01481985333020508
    doppler_rest=1420405000.0 Hz
    doppler_convention=optical)
  [1.39147688e+09 1.39147536e+09 1.39147383e+09 ... 1.39125258e+09
 1.39125105e+09 1.39124953e+09] Hz

In [29]:
ta.plot(xaxis_unit="km/s")

Here the original the spectral axis has changed, albeit by a very small change.

500 Hz or 0.10 km/s

In [30]:
ta.spectral_axis - sa

<Quantity [499.59286833, 499.59232044, 499.59177256, 499.59122467,
           499.59067702, 499.59012914, 499.58958125, 499.58903337,
           499.58848548, 499.58793759, 499.58738995, 499.58684206,
           499.58629417, 499.58574629, 499.5851984 , 499.58465052,
           499.58410263, 499.58355498, 499.5830071 , 499.58245921,
           499.58191133, 499.58136344, 499.58081555, 499.58026791,
           499.57972002, 499.57917213, 499.57862425, 499.57807636,
           499.57752848, 499.57698083, 499.57643294, 499.57588506,
           499.57533717, 499.57478929, 499.5742414 , 499.57369351,
           499.57314587, 499.57259798, 499.57205009, 499.57150221,
           499.57095432, 499.57040644, 499.56985879, 499.5693109 ,
           499.56876302, 499.56821513, 499.56766725, 499.56711936,
           499.56657147, 499.56602383, 499.56547594, 499.56492805,
           499.56438017, 499.56383228, 499.5632844 , 499.56273675,
           499.56218886, 499.56164098, 499.56109309, 499.56054

In [31]:
ta.set_convention('radio')

In [32]:
ta.plot(xaxis_unit="km/s")
print(ta.velocity_axis_to("km/s")[71]) # 6128.47

6128.470305969091 km / s


In [33]:
ta.target

<SkyCoord (FK5: equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
    (206.85210758, -30.40701531, 1000000.)
 (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
    (0., 0., 4386.)>

In [34]:
ta.observer

<GCRS Coordinate (obstime=2005-06-27T02:05:58.000, obsgeoloc=(0., 0., 0.) m, obsgeovel=(0., 0., 0.) m / s): (x, y, z) in m
    (-3418483.37116048, -3651331.51587568, 3945691.32972075)
 (v_x, v_y, v_z) in km / s
    (-4.3110922e-08, 9.51169059e-08, 6.55464828e-08)>

In [35]:
# ta.set_frame('itrs')    # itrs will fail

## Other useful functions

### Spectral Axis Conversion

Convert the spectral axis to any units, frame, and convention with [`Spectrum.velocity_axis_to`](https://dysh.readthedocs.io/en/latest/modules/dysh.spectra.html#dysh.spectra.spectrum.Spectrum.velocity_axis_to). dysh understands some common synonyms like 'heliocentric' for astropy's 'hcrs'.

This returns an array, does not modify the spectrum meta data

? PJT:   why is toframe=   and not  vel_frame=

HeliocentricTrueEcliptic didn't work, but astropy claims it's a frame;   we convert it to lower case?



In [36]:
ta.velocity_axis_to(unit="pc/Myr", toframe='heliocentric', doppler_convention='radio')

<SpectralAxis 
   (observer: <HCRS Coordinate (obstime=2005-06-27T02:05:58.000): (x, y, z) in m
                  (1.44835706e+10, -1.38895619e+11, -6.02107693e+10)
               (v_x, v_y, v_z) in km / s
                  (0., 0., 0.)>
    target: <FK5 Coordinate (equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
                (206.85210758, -30.40701531, 1000000.)
             (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                (0., 0., 4386.)>
    observer to target (computed from above):
      radial_velocity=4386.005213365332 km / s
      redshift=0.014738742242065506
    doppler_rest=1420405000.0 Hz
    doppler_convention=radio)
  [6220.27269497, 6220.60208987, 6220.93148476, 6221.26087966,
   6221.59027455, 6221.91966945, 6222.24906434, 6222.57845924,
   6222.90785413, 6223.23724902, 6223.56664392, 6223.89603881,
   6224.22543371, 6224.5548286 , 6224.8842235 , 6225.21361839,
   6225.54301328, 6225.87240818, 6226.20180307, 6226.5311

In [37]:
# ta.velocity_axis_to(toframe='heliocentricTrueEcliptic', doppler_convention='radio')

In [38]:
ta.velocity_axis_to(unit="km/s")

<SpectralAxis 
   (observer: <GCRS Coordinate (obstime=2005-06-27T02:05:58.000, obsgeoloc=(0., 0., 0.) m, obsgeovel=(0., 0., 0.) m / s): (x, y, z) in m
                  (-3418483.37116048, -3651331.51587568, 3945691.32972075)
               (v_x, v_y, v_z) in km / s
                  (-4.3110922e-08, 9.51169059e-08, 6.55464828e-08)>
    target: <FK5 Coordinate (equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
                (206.85210758, -30.40701531, 1000000.)
             (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                (0., 0., 4386.)>
    observer to target (computed from above):
      radial_velocity=4409.962402228085 km / s
      redshift=0.01481985333020508
    doppler_rest=1420405000.0 Hz
    doppler_convention=radio)
  [6105.60446996, 6105.92652398, 6106.24857801, 6106.57063204,
   6106.89268607, 6107.2147401 , 6107.53679413, 6107.85884815,
   6108.18090218, 6108.50295621, 6108.82501024, 6109.14706427,
   6109.4691183 , 6109.

In [39]:
ta.velocity_axis_to(unit="GHz")

<SpectralAxis 
   (observer: <GCRS Coordinate (obstime=2005-06-27T02:05:58.000, obsgeoloc=(0., 0., 0.) m, obsgeovel=(0., 0., 0.) m / s): (x, y, z) in m
                  (-3418483.37116048, -3651331.51587568, 3945691.32972075)
               (v_x, v_y, v_z) in km / s
                  (-4.3110922e-08, 9.51169059e-08, 6.55464828e-08)>
    target: <FK5 Coordinate (equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
                (206.85210758, -30.40701531, 1000000.)
             (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                (0., 0., 4386.)>
    observer to target (computed from above):
      radial_velocity=4409.962402228085 km / s
      redshift=0.01481985333020508
    doppler_rest=1420405000.0 Hz
    doppler_convention=radio)
  [1.39147688, 1.39147536, 1.39147383, 1.39147231, 1.39147078, 1.39146925,
   1.39146773, 1.3914662 , 1.39146468, 1.39146315, 1.39146162, 1.3914601 ,
   1.39145857, 1.39145705, 1.39145552, 1.391454  , 1.39145247,

In [40]:
ta[70:73]

<Spectrum(flux=<Quantity [0.33822484, 0.59789101, 0.26609857] K> (shape=(3,), mean=0.40074 K); spectral_axis=<SpectralAxis 
   (observer: <ITRS Coordinate (obstime=2005-06-27T02:05:58.000, location=(0.0, 0.0, 0.0) km): (x, y, z) in m
                  (882593.9465029, -4924896.36541728, 3943748.74743984)
               (v_x, v_y, v_z) in km / s
                  (0., 0., 0.)>
    target: <SkyCoord (FK5: equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
                (206.85210758, -30.40701531, 1000000.)
             (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                (0., 0., 4386.)>
    observer to target (computed from above):
      radial_velocity=4410.070039086893 km / s
      redshift=0.014820217767934851
    doppler_rest=1420405000.0 Hz
    doppler_convention=radio)
  [1.39136957e+09 1.39136805e+09 1.39136652e+09] Hz> (length=3))>

In [41]:
ta[70:73].velocity_axis_to(unit="km/s").value[1]

np.float64(6128.575742455784)

### Spectral Shift

Shift a spectrum in place to a given radial velocity or redshift with 
[Spectrum.shift_spectrum_to()](https://specutils.readthedocs.io/en/stable/api/specutils.Spectrum1D.html#specutils.Spectrum1D.shift_spectrum_to)

In [42]:
print(ta.velocity_axis_to("km/s")[71])

6128.470305969091 km / s


In [43]:
print(f"before shift {ta.spectral_axis}")
#ta.shift_spectrum_to(radial_velocity=0*u.km/u.s)
#ta.shift_spectrum_to(radial_velocity=4410.070039086893*u.km/u.s)
ta.shift_spectrum_to(radial_velocity=2000*u.km/u.s)
print(f"after shift {ta.spectral_axis}")

before shift [1.39147688e+09 1.39147536e+09 1.39147383e+09 1.39147231e+09
 1.39147078e+09 1.39146925e+09 1.39146773e+09 1.39146620e+09
 1.39146468e+09 1.39146315e+09 1.39146162e+09 1.39146010e+09
 1.39145857e+09 1.39145705e+09 1.39145552e+09 1.39145400e+09
 1.39145247e+09 1.39145094e+09 1.39144942e+09 1.39144789e+09
 1.39144637e+09 1.39144484e+09 1.39144331e+09 1.39144179e+09
 1.39144026e+09 1.39143874e+09 1.39143721e+09 1.39143568e+09
 1.39143416e+09 1.39143263e+09 1.39143111e+09 1.39142958e+09
 1.39142806e+09 1.39142653e+09 1.39142500e+09 1.39142348e+09
 1.39142195e+09 1.39142043e+09 1.39141890e+09 1.39141737e+09
 1.39141585e+09 1.39141432e+09 1.39141280e+09 1.39141127e+09
 1.39140974e+09 1.39140822e+09 1.39140669e+09 1.39140517e+09
 1.39140364e+09 1.39140212e+09 1.39140059e+09 1.39139906e+09
 1.39139754e+09 1.39139601e+09 1.39139449e+09 1.39139296e+09
 1.39139143e+09 1.39138991e+09 1.39138838e+09 1.39138686e+09
 1.39138533e+09 1.39138380e+09 1.39138228e+09 1.39138075e+09
 1.39137923

In [44]:
ta

<Spectrum(flux=[0.34372934266958427 ... 0.30729395889669986] K (shape=(150,), mean=0.28094 K); spectral_axis=<SpectralAxis 
   (observer: <GCRS Coordinate (obstime=2005-06-27T02:05:58.000, obsgeoloc=(0., 0., 0.) m, obsgeovel=(0., 0., 0.) m / s): (ra, dec, distance) in (deg, deg, m)
                  (226.88638534, 38.2681561, 6370771.57549705)
               (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                  (-33132.93080836, 103576.96454981, -4.73214035e-08)>
    target: <FK5 Coordinate (equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
                (206.85210758, -30.40701531, 1000000.)
             (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                (2.19563458e-18, -2.97325515e-19, 1976.03759777)>
    observer to target (computed from above):
      radial_velocity=1999.9999999922752 km / s
      redshift=0.006693684108879161
    doppler_rest=1420405000.0 Hz
    doppler_convention=radio)
  [1.4027

Changing the redshift?
Here's some confusion, printing the spectrum it says:

doppler_rest  which is spectrum.rest_value   (say 1420 MHz)


PJT  roundoff I guess
old: 4410.070039086893  0.014820217767934851
new: 4410.070039086889  0.014820217767934851




In [45]:
print(ta.velocity_axis_to("km/s")[71])  # 6128.575765276228 

3757.974398543667 km / s


In [46]:
ta.plot(xaxis_unit="km/s")

In [47]:
ta

<Spectrum(flux=[0.34372934266958427 ... 0.30729395889669986] K (shape=(150,), mean=0.28094 K); spectral_axis=<SpectralAxis 
   (observer: <GCRS Coordinate (obstime=2005-06-27T02:05:58.000, obsgeoloc=(0., 0., 0.) m, obsgeovel=(0., 0., 0.) m / s): (ra, dec, distance) in (deg, deg, m)
                  (226.88638534, 38.2681561, 6370771.57549705)
               (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                  (-33132.93080836, 103576.96454981, -4.73214035e-08)>
    target: <FK5 Coordinate (equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
                (206.85210758, -30.40701531, 1000000.)
             (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                (2.19563458e-18, -2.97325515e-19, 1976.03759777)>
    observer to target (computed from above):
      radial_velocity=1999.9999999922752 km / s
      redshift=0.006693684108879161
    doppler_rest=1420405000.0 Hz
    doppler_convention=radio)
  [1.4027

In [48]:
# going back to the original --- fails with target/observer
#ta.set_frame('itrs')   # itrs   fails!!     icrs works
ta.set_frame('icrs')
ta.set_convention('optical')
ta.plot(xaxis_unit="km/s")
print(ta.velocity_axis_to("km/s")[71]) 

3781.413935417491 km / s


In [49]:
# Spectrum.fake_spectrum().set_redshift_to(0.1)
# Spectrum.fake_spectrum().set_radial_velocity_to( 4000 * u.km/u.s)

# ValueError: Cannot specify radial velocity or redshift if both target and observer are specified

### Spectral Axis in Wavelengths
The default is angstrom, use [Quantity.to](https://docs.astropy.org/en/stable/api/astropy.units.Quantity.html#astropy.units.Quantity.to) to convert to other units.

In [50]:
ta.wavelength.to('cm')

<SpectralAxis 
   (observer: <ICRS Coordinate: (x, y, z) in m
                  (1.51192456e+10, -1.38756853e+11, -6.01691775e+10)
               (v_x, v_y, v_z) in km / s
                  (0., 0., 0.)>
    target: <FK5 Coordinate (equinox=J2000.000): (ra, dec, distance) in (deg, deg, kpc)
                (206.85210758, -30.40701531, 1000000.)
             (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
                (2.19563458e-18, -2.97325515e-19, 1976.03759777)>
    observer to target (computed from above):
      radial_velocity=1976.0375977719118 km / s
      redshift=0.006613218790137987
    doppler_rest=1420405000.0 Hz
    doppler_convention=optical)
  [21.37068205, 21.37070548, 21.37072892, 21.37075235, 21.37077579,
   21.37079922, 21.37082266, 21.37084609, 21.37086953, 21.37089296,
   21.3709164 , 21.37093983, 21.37096327, 21.3709867 , 21.37101014,
   21.37103357, 21.37105701, 21.37108045, 21.37110388, 21.37112732,
   21.37115075, 21.37117419, 21.371

### Plot in Channels

Also, you can plot the x-axis in channel units

In [51]:
ta.plot(xaxis_unit='chan');

## Final Stats

Finally, at the end we compute some statistics over a spectrum, merely as a checksum if the notebook is reproducible.

In [52]:
ta.check_stats(0.07586939 * u.K)

23:41:44.271 I rms is OK 
